##### Vector Store 

```

PDF / Website / Documents
          ↓
       Chunking
          ↓
    Embedding Model
          ↓
   ┌──────┴──────┐
   ↓             ↓
 FAISS         Chroma
   ↓             ↓
Vector Search / Retrieval
          ↓
      LLM (Ollama)
          ↓
        Answer
        
```

##### Faiss is a facebook ai similarity search 

In [1]:
# Data Loading

from langchain_community.document_loaders import TextLoader

loader=TextLoader("speech.txt") # just initializing the loader object

data=loader.load()

e:\genai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data

[Document(metadata={'source': 'speech.txt'}, page_content='The Clockmaker of Ratnapur\n\nIn a small hill town called Ratnapur, there lived an old clockmaker named Dinkar Rao. His shop stood at the end of Lantern Street, right next to the town\'s only bakery. The shop was tiny, but it was packed with hundreds of clocks: wall clocks, pocket watches, cuckoo clocks, and even a giant grandfather clock that had not worked for forty years.\n\nEveryone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."\n\nOne rainy morning in July, a girl named Meera walked into the shop. She was twelve years old, soaked from head to 

In [3]:
# split the data into chunks using recursive or character split

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splitted_text=splitter.split_documents(data)

In [4]:
splitted_text

[Document(metadata={'source': 'speech.txt'}, page_content="The Clockmaker of Ratnapur\n\nIn a small hill town called Ratnapur, there lived an old clockmaker named Dinkar Rao. His shop stood at the end of Lantern Street, right next to the town's only bakery. The shop was tiny, but it was packed with hundreds of clocks: wall clocks, pocket watches, cuckoo clocks, and even a giant grandfather clock that had not worked for forty years."),
 Document(metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(metadata={'source': 'speech.txt'}, page_conten

In [5]:
# after splitting the data into chunks next is to create vectors 
# means emebdiing and then storing the details into vectore store ->faiss or chroma 

from langchain_community.embeddings import OllamaEmbeddings

embeddings=OllamaEmbeddings(
    model="nomic-embed-text"
)

C:\Users\Azrap\AppData\Local\Temp\ipykernel_17116\3146289032.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings=OllamaEmbeddings(


In [ ]:
#if we dont want to store in vectore db then we use vector=embeddings.embed_query() or embed_document

from langchain_community.vectorstores import FAISS

vectorstore=FAISS.from_documents(
    splitted_text,
    embedding=embeddings
)



```
--- FAISS.from_documents()

This takes:

your Documents
      +
embedding model
      ↓
creates embeddings
      ↓
stores them in a FAISS index
```


this is roughly doing thiss...

```

for document in splitted_text:

    text = document.page_content

    vector = embeddings.embed_query(text)

    # store vector in FAISS

```

In [7]:
vectorstore

In [ ]:
query = "Why did Dinkar keep the old grandfather clock stopped for so many years?"

results = vectorstore.similarity_search(query) # this finds only the similarr pieces of chunks regarding the query

results

[Document(id='fcf9976c-60cc-41c9-9528-0b91a415d91c', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(id='61443d4c-bd51-4855-a653-f99289d4296a', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why would my grandfather hide it

In [9]:
results[0]

Document(id='fcf9976c-60cc-41c9-9528-0b91a415d91c', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."')

In [10]:
results[0].page_content

'Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'

In [11]:
results[0].metadata

{'source': 'speech.txt'}

In [13]:
for i,res in enumerate(results):
    print(f"{i} - {res.page_content}")
    print("*"*40)

0 - Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."
****************************************
1 - "I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."

Meera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why would my grandfather hide it inside a watch?"
****************************************
2 - Dinkar sat down on his old wooden stool and cried. Meera sat next to him and held his hand. After

##### AS A RETRIVER 

so when you define your vector store as retriver then you cann directlyy do .invoke() and this will basically tke your query ocnverts it into vector and searches for similar pieces of chucnkss

In [14]:
retriver=vectorstore.as_retriever()

retriver.invoke("Why did Dinkar keep the old grandfather clock stopped for so many years?")

[Document(id='fcf9976c-60cc-41c9-9528-0b91a415d91c', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
 Document(id='61443d4c-bd51-4855-a653-f99289d4296a', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why would my grandfather hide it

##### similarity search with score

measn you will get similar pieces of chunks along with the scoress and the returned score distance is l2 distance soo the lowers is better

In [16]:
docs_with_score=vectorstore.similarity_search_with_score(
    query
)
docs_with_score

[(Document(id='fcf9976c-60cc-41c9-9528-0b91a415d91c', metadata={'source': 'speech.txt'}, page_content='Everyone in Ratnapur knew Dinkar. He was seventy-two years old, wore round glasses, and always had a pencil behind his ear. Children loved him because he gave them small brass gears as gifts. Adults respected him because he could fix any clock in the world. But there was one clock he never touched, the grandfather clock in the corner of his shop. When people asked why, he would just smile and say, "That one is waiting for the right time."'),
  np.float32(219.03311)),
 (Document(id='61443d4c-bd51-4855-a653-f99289d4296a', metadata={'source': 'speech.txt'}, page_content='"I always thought he was angry with me," Dinkar said quietly. "I stopped the clock that night, and I never started it again. I promised myself I would only wind it when Harish returned."\n\nMeera looked at the numbers on the paper. "Seven, twenty-one, nineteen seventy-four. It is the date you finished the clock. But why 

```

Your question
     ↓
"Why did Dinkar stop the grandfather clock?"
     ↓
Embedding model
     ↓
Query vector
     ↓
FAISS similarity search
     ↓
Most similar document vectors
     ↓
Original text chunks

```

In [17]:
# saving and loading

vectorstore.save_local('faiss_index')